<a href="https://colab.research.google.com/github/2001020057-tech/Yamet-NaiveBayes/blob/main/APLIKASI_KLINIK_YAMET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# ============================================================
# SEL 1R - PEMULIHAN ENVIRONMENT YAMET
# UNTUK GOOGLE COLAB YANG MENGALAMI RESET RUNTIME
# ============================================================

import os
import sys
import glob
import subprocess
import urllib.request
from pathlib import Path

print("=" * 70)
print("PEMULIHAN ENVIRONMENT PROYEK YAMET")
print("=" * 70)

print("Python:", sys.version.split()[0])

# ============================================================
# 1. MENGHUBUNGKAN GOOGLE DRIVE
# ============================================================

print("\n[1] Memeriksa Google Drive...")

PROJECT_NAME = "Proyek_Naive_Bayes_Yamet"

drive_roots = [
    "/content/drive_yamet_deploy_lg7aivx4/MyDrive",
    "/content/drive/MyDrive"
]

PROJECT_DIR = None

for root in drive_roots + glob.glob("/content/drive*/MyDrive"):

    candidate = os.path.join(root, PROJECT_NAME)

    if os.path.isdir(candidate):
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:

    from google.colab import drive

    DRIVE_MOUNT = "/content/drive_yamet_deploy_lg7aivx4"

    drive.mount(DRIVE_MOUNT)

    PROJECT_DIR = os.path.join(
        DRIVE_MOUNT,
        "MyDrive",
        PROJECT_NAME
    )

if not os.path.isdir(PROJECT_DIR):

    raise FileNotFoundError(
        "Folder proyek Yamet tidak ditemukan: "
        + PROJECT_DIR
    )

print("[OK] Google Drive terhubung.")
print("Lokasi proyek:", PROJECT_DIR)

# ============================================================
# 2. MEMERIKSA DELAPAN ARTEFAK MODEL
# ============================================================

print("\n[2] Memeriksa artefak model...")

ARTIFACT_DIR = os.path.join(
    PROJECT_DIR,
    "artifacts_cnb_streamlit"
)

required_files = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]

if not os.path.isdir(ARTIFACT_DIR):

    raise FileNotFoundError(
        "Folder artefak tidak ditemukan."
    )

for filename in required_files:

    path = os.path.join(
        ARTIFACT_DIR,
        filename
    )

    if not os.path.isfile(path):

        raise FileNotFoundError(
            "Artefak tidak ditemukan: " + filename
        )

    print("[OK]", filename)

print("[OK] Delapan artefak tetap tersedia di Drive.")

# ============================================================
# 3. MEMBUAT VIRTUAL ENVIRONMENT
# ============================================================

print("\n[3] Membuat virtual environment...")

VENV_DIR = "/content/yamet_venv"
VENV_PYTHON = os.path.join(
    VENV_DIR,
    "bin",
    "python"
)

if not os.path.isfile(VENV_PYTHON):

    subprocess.run(
        [
            sys.executable,
            "-m",
            "venv",
            "--without-pip",
            VENV_DIR
        ],
        check=True,
        timeout=120
    )

print("[OK] Virtual environment tersedia.")

# ============================================================
# 4. MEMASANG PIP
# ============================================================

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"

print("\n[4] Memasang pip...")

get_pip_path = "/content/get-pip.py"

request = urllib.request.Request(
    "https://bootstrap.pypa.io/get-pip.py",
    headers={"User-Agent": "Mozilla/5.0"}
)

with urllib.request.urlopen(
    request,
    timeout=120
) as response:

    content = response.read()

if len(content) < 1000:

    raise RuntimeError(
        "File get-pip.py tidak valid."
    )

Path(get_pip_path).write_bytes(content)

subprocess.run(
    [
        VENV_PYTHON,
        get_pip_path
    ],
    check=True,
    env=ENV,
    timeout=300
)

print("[OK] Pip berhasil dipasang.")

# ============================================================
# 5. MEMASANG LIBRARY PROYEK
# ============================================================

print("\n[5] Memasang library machine learning...")

PACKAGES = [
    "numpy==2.1.3",
    "scipy==1.15.3",
    "pandas==2.2.3",
    "scikit-learn==1.8.0",
    "matplotlib==3.10.1",
    "seaborn==0.13.2",
    "joblib==1.4.2",
    "threadpoolctl==3.6.0",
    "narwhals>=2.0.1",
    "streamlit",
    "pyngrok"
]

subprocess.run(
    [
        VENV_PYTHON,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--prefer-binary"
    ] + PACKAGES,
    check=True,
    env=ENV,
    timeout=1200
)

print("[OK] Library berhasil dipasang.")

# ============================================================
# 6. MEMPERBAIKI BACKEND MATPLOTLIB
# ============================================================

print("\n[6] Mengatur backend Matplotlib...")

MPLCONFIGDIR = "/content/matplotlib_config_yamet"

os.makedirs(
    MPLCONFIGDIR,
    exist_ok=True
)

ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = MPLCONFIGDIR

# ============================================================
# 7. MENGUJI SELURUH LIBRARY
# ============================================================

print("\n[7] Menguji library...")

CHECK_CODE = r"""
import sys
import numpy
import scipy
import pandas
import sklearn
import matplotlib
import seaborn
import joblib
import streamlit
import pyngrok

from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-Learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Backend:", matplotlib.get_backend())
print("Seaborn:", seaborn.__version__)
print("Streamlit:", streamlit.__version__)
print("Pyngrok: tersedia")

print("SEMUA IMPORT BERHASIL.")
"""

result = subprocess.run(
    [
        VENV_PYTHON,
        "-c",
        CHECK_CODE
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=90
)

print(result.stdout)

if result.stderr:
    print(result.stderr[-3000:])

if result.returncode != 0:

    raise RuntimeError(
        "Pengujian library gagal."
    )

# ============================================================
# 8. MENYIMPAN KONFIGURASI
# ============================================================

os.environ["YAMET_VENV_DIR"] = VENV_DIR
os.environ["YAMET_VENV_PYTHON"] = VENV_PYTHON

os.environ["YAMET_ARTIFACT_DIR"] = ARTIFACT_DIR
os.environ["YAMET_DATASET_PATH"] = os.path.join(
    PROJECT_DIR,
    "dataset_final_harmonized_v2.csv"
)

os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = MPLCONFIGDIR

# ============================================================
# 9. MEMERIKSA APP.PY
# ============================================================

print("\n[8] Memeriksa aplikasi Streamlit...")

if os.path.isfile("/content/app.py"):

    print("[OK] app.py masih tersedia.")

else:

    print("[INFO] app.py perlu dibuat kembali.")
    print("Gunakan kode SEL 3A dari notebook.")

# ============================================================
# 10. RINGKASAN
# ============================================================

print("\n" + "=" * 70)
print("PEMULIHAN ENVIRONMENT YAMET BERHASIL")
print("=" * 70)

print("Python virtual :", VENV_PYTHON)
print("Folder proyek  :", PROJECT_DIR)
print("Folder artefak :", ARTIFACT_DIR)
print("Backend        : Agg")

print("\nDelapan artefak model berhasil diperiksa.")
print("Environment siap digunakan kembali.")


PEMULIHAN ENVIRONMENT PROYEK YAMET
Python: 3.13.18

[1] Memeriksa Google Drive...
[OK] Google Drive terhubung.
Lokasi proyek: /content/drive/MyDrive/Proyek_Naive_Bayes_Yamet

[2] Memeriksa artefak model...
[OK] model_cnb.pkl
[OK] ordinal_encoder.pkl
[OK] feature_columns.pkl
[OK] feature_options.pkl
[OK] evaluation_results.pkl
[OK] dataset_distribution.pkl
[OK] model_metadata.pkl
[OK] imputation_metadata.pkl
[OK] Delapan artefak tetap tersedia di Drive.

[3] Membuat virtual environment...
[OK] Virtual environment tersedia.

[4] Memasang pip...
[OK] Pip berhasil dipasang.

[5] Memasang library machine learning...
[OK] Library berhasil dipasang.

[6] Mengatur backend Matplotlib...

[7] Menguji library...
Python: 3.13.18
NumPy: 2.1.3
SciPy: 1.15.3
Pandas: 2.2.3
Scikit-Learn: 1.8.0
Matplotlib: 3.10.1
Backend: Agg
Seaborn: 0.13.2
Streamlit: 1.67.0
Pyngrok: tersedia
SEMUA IMPORT BERHASIL.


[8] Memeriksa aplikasi Streamlit...
[OK] app.py masih tersedia.

PEMULIHAN ENVIRONMENT YAMET BERHASIL
P

In [26]:
# ============================================================
# SEL 1A - DIAGNOSTIK VIRTUAL ENVIRONMENT YAMET
# TANPA INSTALASI ULANG
# ============================================================

import os
import subprocess

VENV_PYTHON = "/content/yamet_venv/bin/python"

ENV = os.environ.copy()
ENV["PYTHONNOUSERSITE"] = "1"
ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

print("=" * 70)
print("DIAGNOSTIK LIBRARY VIRTUAL ENVIRONMENT YAMET")
print("=" * 70)

if not os.path.isfile(VENV_PYTHON):
    raise FileNotFoundError("Python virtual environment tidak ditemukan.")

tests = {
    "Python": "import sys; print(sys.version)",
    "NumPy": "import numpy; print(numpy.__version__)",
    "SciPy": "import scipy; print(scipy.__version__)",
    "Pandas": "import pandas; print(pandas.__version__)",
    "Scikit-Learn": "import sklearn; print(sklearn.__version__)",
    "CategoricalNB": "from sklearn.naive_bayes import CategoricalNB; print('OK')",
    "OrdinalEncoder": "from sklearn.preprocessing import OrdinalEncoder; print('OK')",
    "Matplotlib": "import matplotlib; print(matplotlib.__version__)",
    "Seaborn": "import seaborn; print(seaborn.__version__)",
    "Joblib": "import joblib; print(joblib.__version__)",
    "Streamlit": "import streamlit; print(streamlit.__version__)",
    "Pyngrok": "import pyngrok; print('OK')"
}

failed = []

for name, code in tests.items():

    result = subprocess.run(
        [VENV_PYTHON, "-c", code],
        capture_output=True,
        text=True,
        env=ENV,
        timeout=60
    )

    if result.returncode == 0:

        print(f"[OK] {name}: {result.stdout.strip()}")

    else:

        failed.append(name)

        print(f"\n[ERROR] {name}")
        print("STDOUT:")
        print(result.stdout[-1500:])
        print("STDERR:")
        print(result.stderr[-4000:])

print("\n" + "=" * 70)
print("PEMERIKSAAN KETERGANTUNGAN PIP")
print("=" * 70)

pip_check = subprocess.run(
    [VENV_PYTHON, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=60
)

print(pip_check.stdout)
if pip_check.stderr:
    print(pip_check.stderr)

print("\n" + "=" * 70)
print("RINGKASAN")
print("=" * 70)

if failed:
    print("Library bermasalah:", failed)
    print("Perbaikan perlu difokuskan pada library tersebut.")
else:
    print("Seluruh library berhasil diimpor.")

print("DIAGNOSTIK SELESAI.")


DIAGNOSTIK LIBRARY VIRTUAL ENVIRONMENT YAMET
[OK] Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
[OK] NumPy: 2.1.3
[OK] SciPy: 1.15.3
[OK] Pandas: 2.2.3
[OK] Scikit-Learn: 1.8.0
[OK] CategoricalNB: OK
[OK] OrdinalEncoder: OK
[OK] Matplotlib: 3.10.1
[OK] Seaborn: 0.13.2
[OK] Joblib: 1.4.2
[OK] Streamlit: 1.67.0
[OK] Pyngrok: OK

PEMERIKSAAN KETERGANTUNGAN PIP
No broken requirements found.


RINGKASAN
Seluruh library berhasil diimpor.
DIAGNOSTIK SELESAI.


In [27]:

# ============================================================
# SEL 1B - PERBAIKAN BACKEND MATPLOTLIB YAMET
# TIDAK MELAKUKAN INSTALASI ULANG
# ============================================================

import os
import subprocess
from pathlib import Path

print("=" * 70)
print("PERBAIKAN MATPLOTLIB DAN SEABORN")
print("=" * 70)

VENV_PYTHON = "/content/yamet_venv/bin/python"

# ------------------------------------------------------------
# 1. Mengatur backend Matplotlib
# ------------------------------------------------------------

os.environ["MPLBACKEND"] = "Agg"

os.environ["MPLCONFIGDIR"] = (
    "/content/matplotlib_config_yamet"
)

Path(
    os.environ["MPLCONFIGDIR"]
).mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 2. Menyiapkan environment virtual
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = "/content/matplotlib_config_yamet"

print("[OK] Backend Matplotlib:", ENV["MPLBACKEND"])

# ------------------------------------------------------------
# 3. Menguji Matplotlib dan Seaborn
# ------------------------------------------------------------

TEST_CODE = r"""
import os
import numpy as np
import matplotlib

print("Matplotlib:", matplotlib.__version__)
print("Backend:", matplotlib.get_backend())

import matplotlib.pyplot as plt
import seaborn as sns

print("Seaborn:", sns.__version__)

# Menguji pembuatan grafik
fig, ax = plt.subplots(figsize=(5, 3))

ax.plot(
    [1, 2, 3, 4],
    [2, 4, 3, 5]
)

ax.set_title("Tes Grafik Yamet")

fig.savefig(
    "/content/tes_matplotlib_yamet.png",
    dpi=100,
    bbox_inches="tight"
)

plt.close(fig)

print("[OK] Grafik Matplotlib berhasil dibuat.")

# Menguji Seaborn
fig, ax = plt.subplots(figsize=(5, 3))

sns.barplot(
    x=["ASD", "ADHD", "Language"],
    y=[37, 17, 12],
    ax=ax
)

ax.set_title("Tes Grafik Seaborn")

fig.savefig(
    "/content/tes_seaborn_yamet.png",
    dpi=100,
    bbox_inches="tight"
)

plt.close(fig)

print("[OK] Grafik Seaborn berhasil dibuat.")
print("SEMUA PENGUJIAN BERHASIL.")
"""

result = subprocess.run(
    [
        VENV_PYTHON,
        "-c",
        TEST_CODE
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=90
)

print("\nHASIL PENGUJIAN:")
print(result.stdout)

if result.stderr:
    print("\nPESAN STDERR:")
    print(result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError(
        "Pengujian Matplotlib/Seaborn gagal. "
        "Periksa pesan error di atas."
    )

# ------------------------------------------------------------
# 4. Menyimpan konfigurasi untuk sel selanjutnya
# ------------------------------------------------------------

os.environ["YAMET_MPLBACKEND"] = "Agg"

print("\n" + "=" * 70)
print("SEL 1B BERHASIL")
print("=" * 70)

print("[OK] Matplotlib berfungsi.")
print("[OK] Seaborn berfungsi.")
print("[OK] Backend Agg sudah disiapkan.")
print("[OK] Tidak perlu instalasi ulang library.")


PERBAIKAN MATPLOTLIB DAN SEABORN
[OK] Backend Matplotlib: Agg

HASIL PENGUJIAN:
Matplotlib: 3.10.1
Backend: Agg
Seaborn: 0.13.2
[OK] Grafik Matplotlib berhasil dibuat.
[OK] Grafik Seaborn berhasil dibuat.
SEMUA PENGUJIAN BERHASIL.


SEL 1B BERHASIL
[OK] Matplotlib berfungsi.
[OK] Seaborn berfungsi.
[OK] Backend Agg sudah disiapkan.
[OK] Tidak perlu instalasi ulang library.


In [28]:
# ============================================================
# PERBAIKAN SEL 2 - NAMEERROR FEATURE_COLUMNS
# ============================================================

import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

print("=" * 70)
print("PERBAIKAN SCRIPT TRAINING YAMET")
print("=" * 70)

# ------------------------------------------------------------
# 1. KONFIGURASI
# ------------------------------------------------------------

VENV_PYTHON = "/content/yamet_venv/bin/python"

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Proyek_Naive_Bayes_Yamet"
)

SCRIPT_LOCAL = Path(
    "/content/yamet_train.py"
)

SCRIPT_DRIVE = Path(
    PROJECT_DIR
) / "training_scripts" / "yamet_train.py"

DATASET_PATH = os.path.join(
    PROJECT_DIR,
    "dataset_final_harmonized_v2.csv"
)

ARTIFACT_DIR = os.path.join(
    PROJECT_DIR,
    "artifacts_cnb_streamlit"
)

STAGING_DIR = "/content/yamet_artifacts_staging"

MPLCONFIGDIR = "/content/matplotlib_config_yamet"

# ------------------------------------------------------------
# 2. VALIDASI AWAL
# ------------------------------------------------------------

if not os.path.isfile(VENV_PYTHON):
    raise FileNotFoundError(
        "Python virtual tidak ditemukan."
    )

if not os.path.isfile(DATASET_PATH):
    raise FileNotFoundError(
        "Dataset final tidak ditemukan."
    )

if not SCRIPT_LOCAL.exists() and not SCRIPT_DRIVE.exists():
    raise FileNotFoundError(
        "yamet_train.py tidak ditemukan di /content "
        "maupun Google Drive."
    )

print("[OK] Python virtual ditemukan.")
print("[OK] Dataset ditemukan.")

# ------------------------------------------------------------
# 3. MEMBACA SCRIPT TRAINING
# ------------------------------------------------------------

source_path = (
    SCRIPT_LOCAL
    if SCRIPT_LOCAL.exists()
    else SCRIPT_DRIVE
)

script = source_path.read_text(
    encoding="utf-8"
)

old_line = "assert len(feature_columns) == 18"
new_line = "assert len(FEATURE_COLUMNS) == 18"

if old_line in script:

    script = script.replace(
        old_line,
        new_line
    )

    print("[OK] Nama variabel berhasil diperbaiki.")

elif new_line in script:

    print("[INFO] Perbaikan sudah diterapkan.")

else:

    raise RuntimeError(
        "Baris validasi feature_columns tidak ditemukan. "
        "Periksa script training."
    )

# ------------------------------------------------------------
# 4. MENYIMPAN SCRIPT YANG SUDAH DIPERBAIKI
# ------------------------------------------------------------

SCRIPT_LOCAL.parent.mkdir(
    parents=True,
    exist_ok=True
)

SCRIPT_DRIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

SCRIPT_LOCAL.write_text(
    script,
    encoding="utf-8"
)

SCRIPT_DRIVE.write_text(
    script,
    encoding="utf-8"
)

print("[OK] Script sementara diperbarui.")
print("[OK] Script Google Drive diperbarui.")

# ------------------------------------------------------------
# 5. MENYIAPKAN ENVIRONMENT
# ------------------------------------------------------------

os.makedirs(
    MPLCONFIGDIR,
    exist_ok=True
)

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = MPLCONFIGDIR

ENV["YAMET_DATASET_PATH"] = DATASET_PATH
ENV["YAMET_ARTIFACT_DIR"] = ARTIFACT_DIR
ENV["YAMET_STAGING_DIR"] = STAGING_DIR

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

ENV["YAMET_BACKUP_DIR"] = os.path.join(
    PROJECT_DIR,
    "artifacts_cnb_streamlit_backup_" + timestamp
)

# ------------------------------------------------------------
# 6. MENJALANKAN ULANG TRAINING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MENJALANKAN ULANG TRAINING")
print("=" * 70)

result = subprocess.run(
    [
        VENV_PYTHON,
        str(SCRIPT_LOCAL)
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=600
)

print("\nOUTPUT TRAINING:")
print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError(
        "Training masih mengalami error. "
        "Periksa pesan di atas."
    )

# ------------------------------------------------------------
# 7. MEMERIKSA DELAPAN ARTEFAK GOOGLE DRIVE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VALIDASI DELAPAN ARTEFAK")
print("=" * 70)

required_files = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]

for filename in required_files:

    path = os.path.join(
        ARTIFACT_DIR,
        filename
    )

    if not os.path.isfile(path):
        raise FileNotFoundError(
            "Artefak tidak ditemukan: " + filename
        )

    print(
        "[OK]",
        filename,
        f"({os.path.getsize(path):,} bytes)"
    )

# ------------------------------------------------------------
# 8. RINGKASAN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PERBAIKAN SEL 2 BERHASIL")
print("=" * 70)

print("Script training       : Diperbaiki")
print("Dataset               : 67 sampel")
print("Fitur                 : 18")
print("Model                 : Categorical Naive Bayes")
print("Alpha                 : 1.0")
print("Akurasi                : 71.43%")
print("Delapan artefak       : Tersedia")
print("Script di Google Drive: Tersimpan")

print("\nSelanjutnya lanjut ke SEL 3B untuk validasi aplikasi.")

PERBAIKAN SCRIPT TRAINING YAMET
[OK] Python virtual ditemukan.
[OK] Dataset ditemukan.
[INFO] Perbaikan sudah diterapkan.
[OK] Script sementara diperbarui.
[OK] Script Google Drive diperbarui.

MENJALANKAN ULANG TRAINING

OUTPUT TRAINING:
SCRIPT TRAINING CATEGORICAL NAIVE BAYES
Python       : 3.13.18
Scikit-learn : 1.8.0

[B] MEMBACA DATASET
Jumlah sampel: 67
Jumlah kolom : 40

[C] VALIDASI KOLOM SUMBER
[OK] Seluruh kolom sumber tersedia.

[D] DISKRETISASI USIA
Distribusi kelompok usia: {'Remaja_Dewasa': 25, 'Balita': 24, 'Anak-anak': 18}
[OK] Kelompok usia berhasil dibentuk.

[E] VALIDASI FITUR DAN TARGET
[OK] Tidak ada missing value pada 18 fitur.
[OK] Empat kelas diagnosis ditemukan.

[F] DISTRIBUSI DATASET
Attention-Deficit/Hyperactivity Disorder: 17
Autism Spectrum Disorder: 37
Kondisi Perkembangan Lain: 1
Language Disorder: 12

[G] PEMBAGIAN DATA
Data training: 53
Data testing : 14

Distribusi training:
{'Attention-Deficit/Hyperactivity Disorder': 12, 'Autism Spectrum Disorder': 

In [29]:

%%writefile /content/app.py

import os
import glob
from datetime import datetime
from zoneinfo import ZoneInfo

import streamlit as st
import pandas as pd
import numpy as np
import joblib


# ============================================================
# 1. KONFIGURASI APLIKASI
# ============================================================

os.environ["MPLBACKEND"] = "Agg"

st.set_page_config(
    page_title="Sistem Klasifikasi Yamet",
    page_icon="🧩",
    layout="wide",
    initial_sidebar_state="expanded"
)

JAKARTA_TZ = ZoneInfo("Asia/Jakarta")


# ============================================================
# 2. MENENTUKAN LOKASI ARTEFAK
# ============================================================

ARTIFACT_DIR = os.environ.get(
    "YAMET_ARTIFACT_DIR",
    ""
)

if not ARTIFACT_DIR or not os.path.isdir(ARTIFACT_DIR):

    candidates = glob.glob(
        "/content/drive*/MyDrive/"
        "Proyek_Naive_Bayes_Yamet/"
        "artifacts_cnb_streamlit"
    )

    if candidates:
        ARTIFACT_DIR = candidates[0]


REQUIRED_FILES = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]


if not ARTIFACT_DIR or not os.path.isdir(ARTIFACT_DIR):

    st.error(
        "Folder artefak model tidak ditemukan. "
        "Periksa konfigurasi YAMET_ARTIFACT_DIR."
    )

    st.stop()


missing_files = [
    filename
    for filename in REQUIRED_FILES
    if not os.path.isfile(
        os.path.join(ARTIFACT_DIR, filename)
    )
]

if missing_files:

    st.error(
        "Artefak model belum lengkap: "
        + ", ".join(missing_files)
    )

    st.stop()


# ============================================================
# 3. MEMUAT DELAPAN ARTEFAK MODEL
# ============================================================

@st.cache_resource
def load_artifacts(folder):

    loaded = {}

    for filename in REQUIRED_FILES:

        key = filename.replace(".pkl", "")

        loaded[key] = joblib.load(
            os.path.join(folder, filename)
        )

    return loaded


try:

    artifacts = load_artifacts(ARTIFACT_DIR)

    model = artifacts["model_cnb"]

    encoder = artifacts["ordinal_encoder"]

    feature_columns = artifacts["feature_columns"]

    feature_options = artifacts["feature_options"]

    evaluation_results = artifacts["evaluation_results"]

    dataset_distribution = artifacts["dataset_distribution"]

    model_metadata = artifacts["model_metadata"]

    imputation_metadata = artifacts["imputation_metadata"]

except Exception as error:

    st.error(
        "Gagal membaca artefak model: "
        + str(error)
    )

    st.stop()


# ============================================================
# 4. VALIDASI KOMPATIBILITAS MODEL
# ============================================================

if len(feature_columns) != 18:

    st.error(
        "Jumlah fitur tidak sesuai. "
        f"Ditemukan {len(feature_columns)}, seharusnya 18."
    )

    st.stop()


if len(encoder.categories_) != len(feature_columns):

    st.error(
        "Jumlah kategori encoder tidak sesuai "
        "dengan jumlah fitur."
    )

    st.stop()


if len(model.classes_) != 4:

    st.warning(
        "Jumlah kelas model berbeda dari konfigurasi "
        "empat kelas yang diharapkan."
    )


# ============================================================
# 5. LABEL FITUR
# ============================================================

FEATURE_LABELS = {

    "jenis_kelamin": "Jenis Kelamin",

    "kontak_mata": "Kontak Mata",

    "respon_panggil_nama": "Respons Saat Dipanggil Nama",

    "pemahaman_instruksi_reseptif":
        "Pemahaman Instruksi Reseptif",

    "kemampuan_bicara_ekspresif":
        "Kemampuan Bicara Ekspresif",

    "perilaku_repetitif_stimming":
        "Perilaku Repetitif / Stimming",

    "sensory_processing_disorder":
        "Sensory Processing Disorder",

    "hiperaktivitas_impulsivitas":
        "Hiperaktivitas / Impulsivitas",

    "gangguan_fokus_atensi":
        "Gangguan Fokus / Atensi",

    "tantrum_regulasi_emosi":
        "Tantrum / Regulasi Emosi",

    "dispraksia_motorik":
        "Dispraksia Motorik",

    "oromotor_bermasalah":
        "Gangguan Oromotor",

    "denver_personal_sosial":
        "Denver II Personal Sosial",

    "denver_motorik_halus":
        "Denver II Motorik Halus",

    "denver_bahasa":
        "Denver II Bahasa",

    "denver_motorik_kasar":
        "Denver II Motorik Kasar",

    "kemandirian_adl":
        "Kemandirian ADL",

    "kelompok_usia":
        "Kelompok Usia"
}


# ============================================================
# 6. SESSION STATE
# ============================================================

if "yamet_history" not in st.session_state:

    st.session_state["yamet_history"] = []


if "yamet_prediction" not in st.session_state:

    st.session_state["yamet_prediction"] = None


# ============================================================
# 7. FUNGSI RESET FORMULIR
# ============================================================

def reset_prediction_form():

    widget_keys = [
        "input_usia_bulan"
    ]

    widget_keys += [
        "input_" + feature
        for feature in feature_columns
        if feature != "kelompok_usia"
    ]

    for key in widget_keys:

        st.session_state.pop(
            key,
            None
        )

    st.session_state["yamet_prediction"] = None

    st.session_state["yamet_notice"] = (
        "Formulir berhasil dikosongkan. "
        "Silakan masukkan data baru."
    )


# ============================================================
# 8. FUNGSI MEMBUAT DATA RIWAYAT
# ============================================================

def record_to_export_row(record):

    row = {

        "Nomor": record["id"],

        "Waktu Prediksi": record["waktu"],

        "Usia Bulan": record["usia_bulan"],

        "Kelompok Usia": record["kelompok_usia"],

        "Hasil Prediksi": record["prediction"]
    }

    for feature, value in record["input"].items():

        row[
            "Input - " + feature
        ] = value

    for class_name, probability in record["probabilities"].items():

        row[
            "Probabilitas - " + class_name
        ] = probability

    return row


def make_history_dataframe(records):

    rows = [
        record_to_export_row(record)
        for record in records
    ]

    return pd.DataFrame(rows)


# ============================================================
# 9. SIDEBAR
# ============================================================

with st.sidebar:

    st.title("🧩 Yamet")

    st.caption(
        "Sistem Klasifikasi Data Anak Berkebutuhan Khusus"
    )

    st.divider()

    st.subheader("Informasi Model")

    st.write(
        "**Algoritma:** Categorical Naive Bayes"
    )

    st.write(
        "**Alpha:**",
        model_metadata.get("alpha", 1.0)
    )

    st.write(
        "**Jumlah Data:**",
        model_metadata.get("total_samples", 0)
    )

    st.write(
        "**Jumlah Fitur:**",
        model_metadata.get("n_features", 18)
    )

    st.write(
        "**Jumlah Kelas:**",
        len(model.classes_)
    )

    st.divider()

    st.metric(
        "Riwayat Prediksi",
        len(st.session_state["yamet_history"])
    )

    st.divider()

    st.caption(
        "Hasil klasifikasi merupakan prediksi model "
        "machine learning dan bukan diagnosis klinis."
    )


# ============================================================
# 10. HEADER
# ============================================================

st.title(
    "Sistem Klasifikasi Data Anak Berkebutuhan Khusus"
)

st.subheader(
    "Yamet Tanjungpinang"
)

st.write(
    "Aplikasi ini menggunakan algoritma "
    "**Categorical Naive Bayes** untuk melakukan "
    "klasifikasi berdasarkan 18 fitur prediktor."
)

st.warning(
    "Hasil aplikasi merupakan prediksi kelas berdasarkan "
    "model machine learning. Hasil ini bukan diagnosis "
    "medis dan tidak menggantikan penilaian profesional."
)

st.divider()


# ============================================================
# 11. MENU APLIKASI
# ============================================================

tab_prediksi, tab_evaluasi, tab_dataset, tab_riwayat, tab_informasi = st.tabs([

    "🔎 Prediksi",

    "📊 Evaluasi Model",

    "📁 Distribusi Dataset",

    "🕘 Riwayat Prediksi",

    "ℹ️ Informasi Model"

])


# ============================================================
# TAB 1 - PREDIKSI
# ============================================================

with tab_prediksi:

    st.header("Prediksi Kelas Data")

    st.write(
        "Masukkan karakteristik data berdasarkan pilihan "
        "kategori yang tersedia."
    )

    st.info(
        "Usia dimasukkan dalam bulan. "
        "Sistem akan menentukan kelompok usia secara otomatis."
    )

    # --------------------------------------------------------
    # TOMBOL RESET FORMULIR
    # --------------------------------------------------------

    reset_col, info_col = st.columns([1, 3])

    with reset_col:

        st.button(
            "🔄 Reset Formulir",
            on_click=reset_prediction_form,
            use_container_width=True,
            key="reset_form_yamet"
        )

    notice = st.session_state.pop(
        "yamet_notice",
        None
    )

    if notice:

        st.success(notice)

    # --------------------------------------------------------
    # FORM INPUT
    # --------------------------------------------------------

    with st.form("form_prediksi_yamet"):

        st.subheader("Identitas Usia")

        usia_bulan = st.number_input(

            "Usia dalam Bulan",

            min_value=0,

            max_value=1200,

            value=60,

            step=1,

            key="input_usia_bulan",

            help="Masukkan usia dalam satuan bulan."
        )

        st.divider()

        st.subheader("Karakteristik Perkembangan")

        input_values = {}

        input_features = [

            feature
            for feature in feature_columns
            if feature != "kelompok_usia"

        ]

        left_col, right_col = st.columns(2)

        for index, feature in enumerate(input_features):

            target_col = (
                left_col
                if index % 2 == 0
                else right_col
            )

            with target_col:

                options = feature_options.get(
                    feature,
                    []
                )

                if not options:

                    feature_index = feature_columns.index(
                        feature
                    )

                    options = [
                        str(value)
                        for value in encoder.categories_[feature_index]
                    ]

                # Hanya kategori yang tersedia dalam data training.
                options = [

                    str(value)
                    for value in options

                    if str(value).strip().lower()
                    not in [
                        "",
                        "nan",
                        "none",
                        "null",
                        "missing"
                    ]

                ]

                options = list(
                    dict.fromkeys(options)
                )

                if not options:

                    st.error(
                        f"Pilihan kategori {feature} kosong."
                    )

                    st.stop()

                label = FEATURE_LABELS.get(
                    feature,
                    feature.replace("_", " ").title()
                )

                input_values[feature] = st.selectbox(

                    label,

                    options=options,

                    key="input_" + feature

                )

        st.divider()

        submitted = st.form_submit_button(

            "🔎 Proses Klasifikasi",

            use_container_width=True

        )

    # --------------------------------------------------------
    # PROSES PREDIKSI
    # --------------------------------------------------------

    if submitted:

        # Menentukan kelompok usia.
        if usia_bulan <= 60:

            kelompok_usia = "Balita"

        elif usia_bulan <= 144:

            kelompok_usia = "Anak-anak"

        else:

            kelompok_usia = "Remaja_Dewasa"

        input_values["kelompok_usia"] = kelompok_usia

        input_df = pd.DataFrame([

            {
                feature: input_values[feature]
                for feature in feature_columns
            }

        ])

        try:

            # Encoding sesuai proses training.
            encoded = encoder.transform(
                input_df[feature_columns]
            )

            # Kategori dikenal: 1 sampai K.
            # Kategori tidak dikenal: 0.

            encoded = np.where(

                encoded < 0,

                0,

                encoded + 1

            ).astype(np.int64)

            prediction = model.predict(
                encoded
            )[0]

            probabilities = model.predict_proba(
                encoded
            )[0]

            classes = [
                str(value)
                for value in model.classes_
            ]

            probability_dict = {

                class_name: float(probability)

                for class_name, probability
                in zip(classes, probabilities)

            }

            prediction_time = datetime.now(
                JAKARTA_TZ
            ).strftime("%Y-%m-%d %H:%M:%S")

            # Membentuk data riwayat.
            history_record = {

                "id": len(
                    st.session_state["yamet_history"]
                ) + 1,

                "waktu": prediction_time,

                "prediction": str(prediction),

                "usia_bulan": int(usia_bulan),

                "kelompok_usia": kelompok_usia,

                "input": input_values.copy(),

                "probabilities": probability_dict

            }

            # Menyimpan hasil terakhir.
            st.session_state["yamet_prediction"] = (
                history_record
            )

            # Menambahkan hasil ke riwayat.
            st.session_state["yamet_history"].append(
                history_record
            )

            st.success(
                "Prediksi berhasil diproses dan "
                "ditambahkan ke riwayat."
            )

        except Exception as error:

            st.error(
                "Terjadi kesalahan saat melakukan prediksi: "
                + str(error)
            )

    # --------------------------------------------------------
    # MENAMPILKAN HASIL PREDIKSI
    # --------------------------------------------------------

    result = st.session_state.get(
        "yamet_prediction"
    )

    if result:

        st.divider()

        st.header("Hasil Klasifikasi")

        st.success(
            "Proses prediksi berhasil dilakukan."
        )

        st.caption(
            "Waktu prediksi: " + result["waktu"]
        )

        st.metric(
            "Kelas Prediksi Model",
            result["prediction"]
        )

        st.write(
            "**Usia:**",
            result["usia_bulan"],
            "bulan"
        )

        st.write(
            "**Kelompok Usia:**",
            result["kelompok_usia"]
        )

        st.subheader(
            "Distribusi Probabilitas Prediksi"
        )

        probability_df = pd.DataFrame({

            "Kelas": list(
                result["probabilities"].keys()
            ),

            "Probabilitas": list(
                result["probabilities"].values()
            )

        })

        probability_df["Persentase"] = (
            probability_df["Probabilitas"] * 100
        )

        probability_df = probability_df.sort_values(
            "Probabilitas",
            ascending=False
        )

        st.dataframe(

            probability_df.style.format({

                "Probabilitas": "{:.6f}",

                "Persentase": "{:.2f}%"

            }),

            use_container_width=True,

            hide_index=True

        )

        st.bar_chart(

            probability_df.set_index(
                "Kelas"
            )[["Persentase"]],

            y_label="Probabilitas (%)"

        )

        st.caption(
            "Probabilitas merupakan keluaran model, "
            "bukan tingkat kepastian diagnosis klinis."
        )

        # ----------------------------------------------------
        # DOWNLOAD HASIL PREDIKSI
        # ----------------------------------------------------

        export_data = record_to_export_row(
            result
        )

        csv_data = pd.DataFrame(
            [export_data]
        ).to_csv(
            index=False
        ).encode("utf-8-sig")

        txt_data = []

        txt_data.append(
            "HASIL KLASIFIKASI DATA YAMET"
        )

        txt_data.append("=" * 50)

        txt_data.append(
            "Waktu Prediksi: " + result["waktu"]
        )

        txt_data.append(
            "Hasil Prediksi: " + result["prediction"]
        )

        txt_data.append(
            "Usia: " + str(result["usia_bulan"]) + " bulan"
        )

        txt_data.append(
            "Kelompok Usia: " + result["kelompok_usia"]
        )

        txt_data.append("\nKARAKTERISTIK INPUT")

        for feature, value in result["input"].items():

            txt_data.append(
                f"{feature}: {value}"
            )

        txt_data.append("\nPROBABILITAS SETIAP KELAS")

        for class_name, probability in result["probabilities"].items():

            txt_data.append(
                f"{class_name}: {probability * 100:.2f}%"
            )

        txt_data.append(
            "\nCatatan: Hasil merupakan prediksi model "
            "dan bukan diagnosis klinis."
        )

        txt_data = "\n".join(txt_data)

        download_col1, download_col2 = st.columns(2)

        with download_col1:

            st.download_button(

                "⬇️ Unduh Hasil CSV",

                data=csv_data,

                file_name="hasil_klasifikasi_yamet.csv",

                mime="text/csv",

                use_container_width=True,

                key="download_prediction_csv"

            )

        with download_col2:

            st.download_button(

                "⬇️ Unduh Hasil TXT",

                data=txt_data,

                file_name="hasil_klasifikasi_yamet.txt",

                mime="text/plain",

                use_container_width=True,

                key="download_prediction_txt"

            )


# ============================================================
# TAB 2 - EVALUASI MODEL
# ============================================================

with tab_evaluasi:

    st.header("Evaluasi Model")

    st.write(
        "Evaluasi menggunakan data testing dari "
        "pembagian dataset 80:20."
    )

    accuracy = evaluation_results.get(
        "accuracy",
        0
    )

    macro_precision = evaluation_results.get(
        "macro_precision",
        0
    )

    macro_recall = evaluation_results.get(
        "macro_recall",
        0
    )

    macro_f1 = evaluation_results.get(
        "macro_f1",
        0
    )

    metric1, metric2 = st.columns(2)

    metric3, metric4 = st.columns(2)

    metric1.metric(
        "Accuracy",
        f"{accuracy * 100:.2f}%"
    )

    metric2.metric(
        "Macro Precision",
        f"{macro_precision * 100:.2f}%"
    )

    metric3.metric(
        "Macro Recall",
        f"{macro_recall * 100:.2f}%"
    )

    metric4.metric(
        "Macro F1-Score",
        f"{macro_f1 * 100:.2f}%"
    )

    st.divider()

    st.subheader("Classification Report")

    report = evaluation_results.get(
        "classification_report",
        {}
    )

    labels = evaluation_results.get(
        "labels",
        [str(value) for value in model.classes_]
    )

    report_rows = []

    for label in labels:

        item = report.get(label, {})

        report_rows.append({

            "Kelas": label,

            "Precision": item.get("precision", 0),

            "Recall": item.get("recall", 0),

            "F1-Score": item.get("f1-score", 0),

            "Support": item.get("support", 0)

        })

    for key in ["macro avg", "weighted avg"]:

        item = report.get(key, {})

        if item:

            report_rows.append({

                "Kelas": key,

                "Precision": item.get("precision", 0),

                "Recall": item.get("recall", 0),

                "F1-Score": item.get("f1-score", 0),

                "Support": item.get("support", 0)

            })

    report_df = pd.DataFrame(report_rows)

    st.dataframe(

        report_df.style.format({

            "Precision": "{:.4f}",

            "Recall": "{:.4f}",

            "F1-Score": "{:.4f}"

        }),

        use_container_width=True,

        hide_index=True

    )

    st.divider()

    st.subheader("Confusion Matrix")

    cm = np.array(
        evaluation_results.get(
            "confusion_matrix",
            []
        )
    )

    if cm.size:

        confusion_df = pd.DataFrame(

            cm,

            index=labels,

            columns=labels

        )

        st.dataframe(
            confusion_df,
            use_container_width=True
        )

        st.caption(
            "Baris menunjukkan kelas aktual, sedangkan "
            "kolom menunjukkan kelas prediksi."
        )

    st.divider()

    st.subheader("Keterbatasan Evaluasi")

    target_distribution = dataset_distribution.get(
        "overall",
        {}
    )

    rare_classes = {

        label: count

        for label, count in target_distribution.items()

        if count <= 1

    }

    if rare_classes:

        st.warning(
            "Terdapat kelas dengan jumlah data sangat terbatas: "
            + str(rare_classes)
        )

    zero_support = [

        row["Kelas"]

        for row in report_rows

        if row["Kelas"] in labels

        and row["Support"] == 0

    ]

    if zero_support:

        st.warning(
            "Kelas berikut tidak memiliki sampel aktual "
            "pada data testing: "
            + ", ".join(zero_support)
        )

    st.info(
        "Evaluasi ini menggunakan satu pembagian training "
        "dan testing. Hasilnya belum menunjukkan kinerja "
        "generalisasi yang stabil."
    )


# ============================================================
# TAB 3 - DISTRIBUSI DATASET
# ============================================================

with tab_dataset:

    st.header("Distribusi Dataset")

    total_samples = dataset_distribution.get(
        "total_samples",
        0
    )

    train_samples = dataset_distribution.get(
        "train_samples",
        0
    )

    test_samples = dataset_distribution.get(
        "test_samples",
        0
    )

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "Total Data",
        total_samples
    )

    col2.metric(
        "Data Training",
        train_samples
    )

    col3.metric(
        "Data Testing",
        test_samples
    )

    st.divider()

    st.subheader("Distribusi Seluruh Kelas")

    overall = dataset_distribution.get(
        "overall",
        {}
    )

    overall_df = pd.DataFrame({

        "Kelas": list(overall.keys()),

        "Jumlah Data": list(overall.values())

    })

    st.dataframe(

        overall_df,

        use_container_width=True,

        hide_index=True

    )

    if not overall_df.empty:

        st.bar_chart(

            overall_df.set_index("Kelas"),

            y="Jumlah Data"

        )

    st.divider()

    st.subheader(
        "Distribusi Data Training dan Testing"
    )

    training = dataset_distribution.get(
        "training",
        {}
    )

    testing = dataset_distribution.get(
        "testing",
        {}
    )

    distribution_compare = pd.DataFrame({

        "Training": training,

        "Testing": testing

    }).fillna(0).astype(int)

    st.dataframe(
        distribution_compare,
        use_container_width=True
    )

    st.bar_chart(
        distribution_compare
    )


# ============================================================
# TAB 4 - RIWAYAT PREDIKSI
# ============================================================

with tab_riwayat:

    st.header("Riwayat Hasil Klasifikasi")

    st.write(
        "Menu ini menyimpan hasil prediksi yang dilakukan "
        "selama sesi aplikasi berlangsung."
    )

    history = st.session_state["yamet_history"]

    st.metric(
        "Total Riwayat Prediksi",
        len(history)
    )

    st.divider()

    # --------------------------------------------------------
    # PENCARIAN RIWAYAT
    # --------------------------------------------------------

    st.subheader("Pencarian Riwayat")

    search_query = st.text_input(

        "Cari berdasarkan hasil prediksi, usia, waktu, atau input",

        placeholder="Contoh: Autism, ADHD, 60, Balita...",

        key="search_history_yamet"

    )

    if not history:

        st.info(
            "Belum terdapat riwayat prediksi. "
            "Silakan lakukan klasifikasi melalui menu Prediksi."
        )

    else:

        history_df = make_history_dataframe(
            list(reversed(history))
        )

        # Filter pencarian pada seluruh kolom.
        if search_query.strip():

            mask = history_df.astype(str).apply(

                lambda row: search_query.casefold()
                in " ".join(row.tolist()).casefold(),

                axis=1

            )

            filtered_df = history_df.loc[mask].copy()

        else:

            filtered_df = history_df.copy()

        st.write(
            f"Jumlah hasil pencarian: {len(filtered_df)}"
        )

        if filtered_df.empty:

            st.warning(
                "Tidak ditemukan riwayat yang sesuai "
                "dengan kata kunci."
            )

        else:

            # Tampilan tabel utama.
            display_columns = [

                "Nomor",

                "Waktu Prediksi",

                "Usia Bulan",

                "Kelompok Usia",

                "Hasil Prediksi"

            ]

            st.dataframe(

                filtered_df[display_columns],

                use_container_width=True,

                hide_index=True

            )

            # ------------------------------------------------
            # DETAIL RIWAYAT
            # ------------------------------------------------

            st.divider()

            st.subheader("Detail Riwayat")

            visible_ids = set(
                filtered_df["Nomor"].tolist()
            )

            filtered_records = [

                record

                for record in reversed(history)

                if record["id"] in visible_ids

            ]

            for record in filtered_records[:10]:

                with st.expander(

                    f"Prediksi #{record['id']} | "
                    f"{record['prediction']} | "
                    f"{record['waktu']}"

                ):

                    st.write(
                        "**Usia:**",
                        record["usia_bulan"],
                        "bulan"
                    )

                    st.write(
                        "**Kelompok Usia:**",
                        record["kelompok_usia"]
                    )

                    st.write("**Karakteristik input:**")

                    input_df = pd.DataFrame(

                        list(record["input"].items()),

                        columns=[
                            "Fitur",
                            "Nilai"
                        ]

                    )

                    st.dataframe(

                        input_df,

                        use_container_width=True,

                        hide_index=True

                    )

                    st.write(
                        "**Probabilitas setiap kelas:**"
                    )

                    probability_df = pd.DataFrame({

                        "Kelas": list(
                            record["probabilities"].keys()
                        ),

                        "Probabilitas (%)": [

                            value * 100

                            for value in record["probabilities"].values()

                        ]

                    })

                    st.dataframe(

                        probability_df.style.format({
                            "Probabilitas (%)": "{:.2f}%"
                        }),

                        use_container_width=True,

                        hide_index=True

                    )

            if len(filtered_records) > 10:

                st.caption(
                    "Sepuluh riwayat terbaru ditampilkan pada bagian detail. "
                    "Seluruh hasil tetap tersedia pada unduhan."
                )

            # ------------------------------------------------
            # UNDUH RIWAYAT CSV
            # ------------------------------------------------

            csv_history = filtered_df.to_csv(
                index=False
            ).encode("utf-8-sig")

            # ------------------------------------------------
            # MEMBUAT LAPORAN TXT
            # ------------------------------------------------

            selected_ids = set(
                filtered_df["Nomor"].tolist()
            )

            report_records = [

                record

                for record in reversed(history)

                if record["id"] in selected_ids

            ]

            txt_lines = [

                "LAPORAN RIWAYAT KLASIFIKASI YAMET",

                "=" * 60,

                "Jumlah hasil: " + str(len(report_records)),

                "Dibuat pada: "
                + datetime.now(JAKARTA_TZ).strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),

                ""

            ]

            for record in report_records:

                txt_lines.append(
                    "=" * 60
                )

                txt_lines.append(
                    "RIWAYAT PREDIKSI #" + str(record["id"])
                )

                txt_lines.append(
                    "Waktu: " + record["waktu"]
                )

                txt_lines.append(
                    "Prediksi: " + record["prediction"]
                )

                txt_lines.append(
                    "Usia: " + str(record["usia_bulan"]) + " bulan"
                )

                txt_lines.append(
                    "Kelompok Usia: " + record["kelompok_usia"]
                )

                txt_lines.append("\nKARAKTERISTIK INPUT:")

                for feature, value in record["input"].items():

                    txt_lines.append(
                        f"{feature}: {value}"
                    )

                txt_lines.append("\nPROBABILITAS:")

                for class_name, value in record["probabilities"].items():

                    txt_lines.append(
                        f"{class_name}: {value * 100:.2f}%"
                    )

                txt_lines.append("")

            txt_lines.append(
                "Catatan: Hasil merupakan prediksi model "
                "dan bukan diagnosis klinis."
            )

            txt_history = "\n".join(txt_lines)

            # ------------------------------------------------
            # TOMBOL DOWNLOAD
            # ------------------------------------------------

            st.divider()

            st.subheader("Unduh Laporan Riwayat")

            download_col1, download_col2 = st.columns(2)

            with download_col1:

                st.download_button(

                    "⬇️ Unduh Riwayat CSV",

                    data=csv_history,

                    file_name="riwayat_klasifikasi_yamet.csv",

                    mime="text/csv",

                    use_container_width=True,

                    key="download_history_csv"

                )

            with download_col2:

                st.download_button(

                    "⬇️ Unduh Laporan TXT",

                    data=txt_history,

                    file_name="laporan_riwayat_yamet.txt",

                    mime="text/plain",

                    use_container_width=True,

                    key="download_history_txt"

                )

    st.divider()

    st.caption(
        "Catatan: Riwayat disimpan sementara di session aplikasi "
        "dan tidak otomatis tersimpan permanen di Google Drive. "
        "Unduh laporan untuk menyimpan salinannya."
    )


# ============================================================
# TAB 5 - INFORMASI MODEL
# ============================================================

with tab_informasi:

    st.header("Informasi Model")

    st.subheader("Konfigurasi Training")

    metadata_table = {

        "Algoritma": model_metadata.get(
            "model_name",
            "Categorical Naive Bayes"
        ),

        "Alpha": model_metadata.get(
            "alpha",
            1.0
        ),

        "Random State": model_metadata.get(
            "random_state",
            42
        ),

        "Test Size": model_metadata.get(
            "test_size",
            0.20
        ),

        "Jumlah Data": model_metadata.get(
            "total_samples",
            0
        ),

        "Jumlah Fitur": model_metadata.get(
            "n_features",
            18
        ),

        "Jumlah Kelas": len(model.classes_),

        "Versi Python": model_metadata.get(
            "python_version",
            "-"
        ),

        "Versi Scikit-Learn": model_metadata.get(
            "sklearn_version",
            "-"
        )

    }

    metadata_df = pd.DataFrame(

        list(metadata_table.items()),

        columns=[
            "Parameter",
            "Nilai"
        ]

    )

    st.dataframe(

        metadata_df,

        use_container_width=True,

        hide_index=True

    )

    st.divider()

    st.subheader("Daftar Fitur Prediktor")

    feature_df = pd.DataFrame({

        "No": range(
            1,
            len(feature_columns) + 1
        ),

        "Nama Fitur": feature_columns,

        "Label": [

            FEATURE_LABELS.get(
                feature,
                feature
            )

            for feature in feature_columns

        ]

    })

    st.dataframe(

        feature_df,

        use_container_width=True,

        hide_index=True

    )

    st.divider()

    st.subheader("Informasi Preprocessing")

    st.write(

        "**Metode:**",

        imputation_metadata.get(
            "method",
            "Tidak tersedia"
        )

    )

    st.write(

        "**Total missing sebelum preprocessing:**",

        imputation_metadata.get(
            "total_missing_before",
            0
        )

    )

    st.write(

        "**Total missing setelah preprocessing:**",

        imputation_metadata.get(
            "total_missing_after",
            0
        )

    )

    st.write(

        "**Imputasi diterapkan:**",

        imputation_metadata.get(
            "imputation_applied",
            False
        )

    )

    feature_metadata = imputation_metadata.get(
        "feature_metadata",
        {}
    )

    if feature_metadata:

        audit_rows = []

        for feature, item in feature_metadata.items():

            audit_rows.append({

                "Fitur": feature,

                "Missing Sebelum": item.get(
                    "missing_before",
                    0
                ),

                "Missing Setelah": item.get(
                    "missing_after",
                    0
                ),

                "Strategi": item.get(
                    "strategy",
                    "-"
                )

            })

        st.dataframe(

            pd.DataFrame(audit_rows),

            use_container_width=True,

            hide_index=True

        )

    st.divider()

    st.subheader("Catatan Penggunaan")

    st.write(
        "Aplikasi ini merupakan implementasi model klasifikasi "
        "untuk kepentingan penelitian."
    )

    st.write(
        "Hasil prediksi tidak menggantikan asesmen, pemeriksaan, "
        "atau keputusan diagnostik oleh profesional."
    )


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "Proyek Klasifikasi Data Diagnosa Anak Berkebutuhan Khusus "
    "| Yamet Tanjungpinang"
)


Overwriting /content/app.py


In [30]:

# ============================================================
# SEL 3B - VALIDASI APLIKASI STREAMLIT YAMET
# ============================================================

import os
import subprocess

VENV_PYTHON = "/content/yamet_venv/bin/python"
APP_PATH = "/content/app.py"

ARTIFACT_DIR = os.environ.get(
    "YAMET_ARTIFACT_DIR",
    "/content/drive_yamet_deploy_lg7aivx4/"
    "MyDrive/Proyek_Naive_Bayes_Yamet/"
    "artifacts_cnb_streamlit"
)

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = "/content/matplotlib_config_yamet"
ENV["YAMET_ARTIFACT_DIR"] = ARTIFACT_DIR

print("=" * 70)
print("VALIDASI APLIKASI STREAMLIT YAMET")
print("=" * 70)

# 1. Validasi file
assert os.path.isfile(APP_PATH), "app.py tidak ditemukan."

print("[OK] app.py ditemukan.")

# 2. Kompilasi sintaks Python
result = subprocess.run(
    [
        VENV_PYTHON,
        "-m",
        "py_compile",
        APP_PATH
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=60
)

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "Sintaks app.py tidak valid."
    )

print("[OK] Sintaks Python valid.")

# 3. Validasi import
IMPORT_CODE = r"""
import streamlit
import pandas
import numpy
import joblib
import sklearn

print("Streamlit:", streamlit.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("Scikit-Learn:", sklearn.__version__)

print("IMPORT BERHASIL.")
"""

result = subprocess.run(
    [
        VENV_PYTHON,
        "-c",
        IMPORT_CODE
    ],
    capture_output=True,
    text=True,
    env=ENV,
    timeout=60
)

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "Import library aplikasi gagal."
    )

print(result.stdout)

# 4. Validasi delapan artefak
required = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]

for filename in required:

    path = os.path.join(
        ARTIFACT_DIR,
        filename
    )

    if not os.path.isfile(path):

        raise FileNotFoundError(
            "Artefak tidak ditemukan: " + filename
        )

    print("[OK]", filename)

print("\n" + "=" * 70)
print("SEL 3B BERHASIL")
print("=" * 70)

print("Aplikasi Streamlit siap untuk deployment.")
print("Lokasi aplikasi:", APP_PATH)
print("Lokasi artefak:", ARTIFACT_DIR)


VALIDASI APLIKASI STREAMLIT YAMET
[OK] app.py ditemukan.
[OK] Sintaks Python valid.
Streamlit: 1.67.0
Pandas: 2.2.3
NumPy: 2.1.3
Scikit-Learn: 1.8.0
IMPORT BERHASIL.

[OK] model_cnb.pkl
[OK] ordinal_encoder.pkl
[OK] feature_columns.pkl
[OK] feature_options.pkl
[OK] evaluation_results.pkl
[OK] dataset_distribution.pkl
[OK] model_metadata.pkl
[OK] imputation_metadata.pkl

SEL 3B BERHASIL
Aplikasi Streamlit siap untuk deployment.
Lokasi aplikasi: /content/app.py
Lokasi artefak: /content/drive/MyDrive/Proyek_Naive_Bayes_Yamet/artifacts_cnb_streamlit


In [31]:

# ============================================================
# SEL 3C - PEMULIHAN DAN RESTART STREAMLIT YAMET
# VERSI OTOMATIS SETELAH RESET RUNTIME
#
# Fungsi:
# 1. Memeriksa app.py.
# 2. Memeriksa delapan artefak.
# 3. Mendeteksi server Streamlit lama.
# 4. Membuat file konfigurasi port otomatis.
# 5. Menjalankan Streamlit pada port tersedia.
# 6. Memeriksa kesehatan server.
# ============================================================

import os
import time
import socket
import signal
import subprocess
import urllib.request
from pathlib import Path

print("=" * 70)
print("PEMULIHAN STREAMLIT YAMET")
print("=" * 70)

# ============================================================
# 1. KONFIGURASI
# ============================================================

PYTHON = "/content/yamet_venv/bin/python"

APP = "/content/app.py"

PORT_FILE = "/content/yamet_streamlit_port.txt"

LOG_FILE = "/content/streamlit_yamet.log"

MPLCONFIGDIR = "/content/matplotlib_config_yamet"

ARTIFACT_DIR = os.environ.get(
    "YAMET_ARTIFACT_DIR",
    ""
)

if not ARTIFACT_DIR or not os.path.isdir(ARTIFACT_DIR):

    candidates = [
        "/content/drive_yamet_deploy_lg7aivx4/MyDrive/"
        "Proyek_Naive_Bayes_Yamet/artifacts_cnb_streamlit",

        "/content/drive/MyDrive/"
        "Proyek_Naive_Bayes_Yamet/artifacts_cnb_streamlit"
    ]

    ARTIFACT_DIR = next(
        (
            path for path in candidates
            if os.path.isdir(path)
        ),
        None
    )

# ============================================================
# 2. VALIDASI FILE
# ============================================================

if not os.path.isfile(PYTHON):
    raise FileNotFoundError(
        "Virtual environment tidak ditemukan. "
        "Jalankan SEL 1R terlebih dahulu."
    )

if not os.path.isfile(APP):
    raise FileNotFoundError(
        "app.py tidak ditemukan. "
        "Jalankan SEL 3A terlebih dahulu."
    )

if not ARTIFACT_DIR:
    raise FileNotFoundError(
        "Folder artefak model tidak ditemukan."
    )

required_files = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]

for filename in required_files:

    if not os.path.isfile(
        os.path.join(ARTIFACT_DIR, filename)
    ):
        raise FileNotFoundError(
            "Artefak tidak ditemukan: " + filename
        )

print("[OK] Python virtual ditemukan.")
print("[OK] app.py ditemukan.")
print("[OK] Delapan artefak tersedia.")

# ============================================================
# 3. MENYIAPKAN ENVIRONMENT
# ============================================================

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"

ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = MPLCONFIGDIR

ENV["YAMET_ARTIFACT_DIR"] = ARTIFACT_DIR

os.makedirs(
    MPLCONFIGDIR,
    exist_ok=True
)

# ============================================================
# 4. FUNGSI PEMERIKSAAN SERVER
# ============================================================

def check_streamlit(port):

    try:

        response = urllib.request.urlopen(
            f"http://127.0.0.1:{port}/_stcore/health",
            timeout=3
        )

        return response.status == 200

    except Exception:

        return False


# ============================================================
# 5. MENDETEKSI STREAMLIT YANG MASIH AKTIF
# ============================================================

print("\n[1] Mendeteksi server Streamlit...")

active_port = None

for port in range(8501, 8511):

    if check_streamlit(port):

        active_port = port

        print(
            "[INFO] Server ditemukan pada port:",
            port
        )

        break

# ============================================================
# 6. MENCARI PORT KOSONG JIKA SERVER BELUM AKTIF
# ============================================================

if active_port is None:

    print("[INFO] Server belum aktif.")
    print("Mencari port kosong...")

    for port in range(8501, 8511):

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        try:

            sock.bind(("127.0.0.1", port))

            active_port = port

            sock.close()

            break

        except OSError:

            sock.close()

    if active_port is None:

        raise RuntimeError(
            "Tidak ditemukan port kosong antara 8501-8510."
        )

    print("[OK] Port tersedia:", active_port)

# ============================================================
# 7. MENGHENTIKAN STREAMLIT LAMA JIKA ADA
# ============================================================

print("\n[2] Memeriksa proses Streamlit lama...")

process_info = subprocess.run(
    ["ps", "-eo", "pid=,args="],
    capture_output=True,
    text=True
)

old_pids = []

for line in process_info.stdout.splitlines():

    parts = line.strip().split(None, 1)

    if len(parts) < 2:
        continue

    try:
        pid = int(parts[0])
    except ValueError:
        continue

    command = parts[1]

    if (
        "streamlit run /content/app.py" in command
        and pid != os.getpid()
    ):
        old_pids.append(pid)

for pid in old_pids:

    try:

        os.kill(pid, signal.SIGTERM)

        print("[OK] Menghentikan Streamlit lama:", pid)

    except ProcessLookupError:
        pass

# Menunggu server lama berhenti.
for _ in range(15):

    if not check_streamlit(active_port):
        break

    time.sleep(1)

if check_streamlit(active_port):

    raise RuntimeError(
        "Server lama belum berhenti. "
        "Periksa proses Streamlit secara manual."
    )

print("[OK] Pemeriksaan proses selesai.")

# ============================================================
# 8. MEMBUAT KEMBALI FILE KONFIGURASI PORT
# ============================================================

Path(PORT_FILE).write_text(
    str(active_port),
    encoding="utf-8"
)

os.environ["YAMET_STREAMLIT_PORT"] = str(active_port)

print("\n[OK] File konfigurasi port dibuat kembali.")
print("Port:", active_port)

# ============================================================
# 9. MENJALANKAN STREAMLIT
# ============================================================

print("\n[3] Menjalankan Streamlit versi terbaru...")

with open(
    LOG_FILE,
    "w",
    buffering=1
) as log:

    process = subprocess.Popen(

        [
            PYTHON,
            "-m",
            "streamlit",
            "run",
            APP,

            "--server.port",
            str(active_port),

            "--server.address",
            "0.0.0.0",

            "--server.headless",
            "true",

            "--server.fileWatcherType",
            "none",

            "--browser.gatherUsageStats",
            "false"
        ],

        stdin=subprocess.DEVNULL,

        stdout=log,

        stderr=subprocess.STDOUT,

        env=ENV,

        start_new_session=True,

        close_fds=True
    )

print("[OK] Proses Streamlit dibuat.")
print("PID:", process.pid)

# ============================================================
# 10. MEMERIKSA KESEHATAN SERVER
# ============================================================

print("\n[4] Memeriksa kesehatan server...")

success = False

for attempt in range(40):

    if check_streamlit(active_port):

        success = True

        break

    if process.poll() is not None:

        print("[ERROR] Proses Streamlit berhenti.")

        break

    time.sleep(1)

if not success:

    print("\nLOG STREAMLIT:")

    if os.path.isfile(LOG_FILE):

        lines = Path(LOG_FILE).read_text(
            errors="replace"
        ).splitlines()

        print("\n".join(lines[-40:]))

    raise RuntimeError(
        "Streamlit gagal dijalankan."
    )

# ============================================================
# 11. RINGKASAN
# ============================================================

print("\n" + "=" * 70)
print("SEL 3C BERHASIL")
print("=" * 70)

print("Status Streamlit : AKTIF")
print("Port             :", active_port)

print(
    "Alamat lokal     :",
    f"http://127.0.0.1:{active_port}"
)

print("Folder artefak   :", ARTIFACT_DIR)
print("Konfigurasi port :", PORT_FILE)

print("\nAplikasi Yamet sudah berjalan kembali.")
print("Selanjutnya periksa koneksi ngrok.")


PEMULIHAN STREAMLIT YAMET
[OK] Python virtual ditemukan.
[OK] app.py ditemukan.
[OK] Delapan artefak tersedia.

[1] Mendeteksi server Streamlit...
[INFO] Server ditemukan pada port: 8501

[2] Memeriksa proses Streamlit lama...
[OK] Menghentikan Streamlit lama: 61947
[OK] Pemeriksaan proses selesai.

[OK] File konfigurasi port dibuat kembali.
Port: 8501

[3] Menjalankan Streamlit versi terbaru...
[OK] Proses Streamlit dibuat.
PID: 63044

[4] Memeriksa kesehatan server...

SEL 3C BERHASIL
Status Streamlit : AKTIF
Port             : 8501
Alamat lokal     : http://127.0.0.1:8501
Folder artefak   : /content/drive/MyDrive/Proyek_Naive_Bayes_Yamet/artifacts_cnb_streamlit
Konfigurasi port : /content/yamet_streamlit_port.txt

Aplikasi Yamet sudah berjalan kembali.
Selanjutnya periksa koneksi ngrok.


In [32]:

# ============================================================
# SEL 4A - MENJALANKAN STREAMLIT YAMET
# PROSES TERPISAH DAN TIDAK MENGUNCI SEL
# ============================================================

import os
import time
import socket
import subprocess
import urllib.request
from pathlib import Path

print("=" * 70)
print("MENJALANKAN STREAMLIT YAMET")
print("=" * 70)

# ------------------------------------------------------------
# 1. KONFIGURASI
# ------------------------------------------------------------

PYTHON = "/content/yamet_venv/bin/python"
APP = "/content/app.py"

ARTIFACT_DIR = os.environ.get(
    "YAMET_ARTIFACT_DIR",
    "/content/drive_yamet_deploy_lg7aivx4/"
    "MyDrive/Proyek_Naive_Bayes_Yamet/"
    "artifacts_cnb_streamlit"
)

LOG_FILE = "/content/streamlit_yamet.log"
PORT_FILE = "/content/yamet_streamlit_port.txt"

MPLCONFIGDIR = "/content/matplotlib_config_yamet"

# ------------------------------------------------------------
# 2. VALIDASI
# ------------------------------------------------------------

assert os.path.isfile(PYTHON), "Python virtual tidak ditemukan."
assert os.path.isfile(APP), "app.py tidak ditemukan."
assert os.path.isdir(ARTIFACT_DIR), "Folder artefak tidak ditemukan."

required = [
    "model_cnb.pkl",
    "ordinal_encoder.pkl",
    "feature_columns.pkl",
    "feature_options.pkl",
    "evaluation_results.pkl",
    "dataset_distribution.pkl",
    "model_metadata.pkl",
    "imputation_metadata.pkl"
]

for filename in required:

    path = os.path.join(ARTIFACT_DIR, filename)

    if not os.path.isfile(path):
        raise FileNotFoundError(filename)

print("[OK] Python virtual ditemukan.")
print("[OK] app.py ditemukan.")
print("[OK] Delapan artefak model ditemukan.")

# ------------------------------------------------------------
# 3. ENVIRONMENT STREAMLIT
# ------------------------------------------------------------

ENV = os.environ.copy()

ENV["PYTHONNOUSERSITE"] = "1"
ENV["PYTHONUNBUFFERED"] = "1"
ENV["MPLBACKEND"] = "Agg"
ENV["MPLCONFIGDIR"] = MPLCONFIGDIR
ENV["YAMET_ARTIFACT_DIR"] = ARTIFACT_DIR

os.makedirs(MPLCONFIGDIR, exist_ok=True)

# ------------------------------------------------------------
# 4. FUNGSI PEMERIKSAAN SERVER
# ------------------------------------------------------------

def check_streamlit(port):

    try:

        url = (
            f"http://127.0.0.1:{port}"
            "/_stcore/health"
        )

        response = urllib.request.urlopen(
            url,
            timeout=2
        )

        return response.status == 200

    except Exception:

        return False


# ------------------------------------------------------------
# 5. MEMERIKSA SERVER YANG SUDAH AKTIF
# ------------------------------------------------------------

print("\n[1] Memeriksa server Streamlit...")

active_port = None

for port in range(8501, 8511):

    if check_streamlit(port):

        active_port = port
        print("[OK] Streamlit sudah aktif pada port:", port)
        break

# ------------------------------------------------------------
# 6. MENCARI PORT KOSONG
# ------------------------------------------------------------

if active_port is None:

    print("[INFO] Server belum aktif.")
    print("Mencari port kosong...")

    selected_port = None

    for port in range(8501, 8511):

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        try:

            sock.bind(("127.0.0.1", port))
            selected_port = port
            sock.close()
            break

        except OSError:

            sock.close()

    if selected_port is None:

        raise RuntimeError(
            "Tidak ditemukan port kosong antara 8501-8510."
        )

    print("[OK] Port tersedia:", selected_port)

    # --------------------------------------------------------
    # 7. MENJALANKAN STREAMLIT SECARA TERPISAH
    # --------------------------------------------------------

    print("\n[2] Menjalankan aplikasi Streamlit...")

    with open(LOG_FILE, "ab", buffering=0) as log:

        process = subprocess.Popen(
            [
                PYTHON,
                "-m",
                "streamlit",
                "run",
                APP,
                "--server.port",
                str(selected_port),
                "--server.address",
                "0.0.0.0",
                "--server.headless",
                "true",
                "--server.fileWatcherType",
                "none",
                "--browser.gatherUsageStats",
                "false"
            ],
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            env=ENV,
            start_new_session=True,
            close_fds=True
        )

    print("[OK] Proses Streamlit dibuat.")
    print("PID:", process.pid)

    active_port = selected_port

    # --------------------------------------------------------
    # 8. MENUNGGU SERVER MAKSIMAL 35 DETIK
    # --------------------------------------------------------

    print("\n[3] Memeriksa kesehatan server...")

    success = False

    for attempt in range(35):

        if check_streamlit(active_port):

            success = True
            break

        if process.poll() is not None:

            print("[ERROR] Proses Streamlit berhenti.")
            break

        time.sleep(1)

    if not success:

        print("\n[ERROR] Streamlit belum berhasil aktif.")

        if os.path.isfile(LOG_FILE):

            lines = Path(LOG_FILE).read_text(
                errors="replace"
            ).splitlines()

            print("\nLOG TERAKHIR:")
            print("\n".join(lines[-40:]))

        raise RuntimeError(
            "Server Streamlit gagal aktif."
        )

    print("[OK] Streamlit berhasil dijalankan.")

# ------------------------------------------------------------
# 9. MENYIMPAN PORT
# ------------------------------------------------------------

if not check_streamlit(active_port):

    raise RuntimeError(
        "Server tidak merespons pada pemeriksaan akhir."
    )

Path(PORT_FILE).write_text(
    str(active_port),
    encoding="utf-8"
)

os.environ["YAMET_STREAMLIT_PORT"] = str(active_port)

print("\n" + "=" * 70)
print("SEL 4A BERHASIL")
print("=" * 70)

print("Status       : AKTIF")
print("Port         :", active_port)
print("Alamat lokal : http://127.0.0.1:" + str(active_port))
print("Log          :", LOG_FILE)

print("\nStreamlit berjalan dalam proses terpisah.")
print("Selanjutnya jalankan SEL 4B untuk menghubungkan ngrok.")


MENJALANKAN STREAMLIT YAMET
[OK] Python virtual ditemukan.
[OK] app.py ditemukan.
[OK] Delapan artefak model ditemukan.

[1] Memeriksa server Streamlit...
[OK] Streamlit sudah aktif pada port: 8501

SEL 4A BERHASIL
Status       : AKTIF
Port         : 8501
Alamat lokal : http://127.0.0.1:8501
Log          : /content/streamlit_yamet.log

Streamlit berjalan dalam proses terpisah.
Selanjutnya jalankan SEL 4B untuk menghubungkan ngrok.


In [33]:
import urllib.request
import json

print("=" * 60)
print("PEMERIKSAAN STREAMLIT DAN NGROK")
print("=" * 60)

# 1. Periksa Streamlit
try:
    response = urllib.request.urlopen(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=5
    )
    print("[OK] Streamlit:", response.read().decode())
except Exception as e:
    print("[ERROR] Streamlit:", e)

# 2. Cari dashboard lokal ngrok
found = False

for port in range(4040, 4046):
    try:
        url = f"http://127.0.0.1:{port}/api/tunnels"

        response = urllib.request.urlopen(url, timeout=2)
        data = json.loads(response.read().decode())

        print(f"\n[OK] Agent ngrok ditemukan pada port {port}")

        tunnels = data.get("tunnels", [])

        if tunnels:
            for tunnel in tunnels:
                print("Public URL :", tunnel.get("public_url"))
                print("Protocol   :", tunnel.get("proto"))
                print("Alamat lokal:",
                      tunnel.get("config", {}).get("addr"))
        else:
            print("[INFO] Belum ada tunnel aktif.")

        found = True
        break

    except Exception:
        continue

if not found:
    print("\n[INFO] Dashboard lokal ngrok belum ditemukan.")

PEMERIKSAAN STREAMLIT DAN NGROK
[OK] Streamlit: ok

[OK] Agent ngrok ditemukan pada port 4040
Public URL : https://splotchy-upstate-freewill.ngrok-free.dev
Protocol   : https
Alamat lokal: http://localhost:8501
